# Iterative Fixer Agent Prompt Training

Train the hybrid pipeline's fixer prompt by iterating on 3 PubTabNet images:
1. **Evaluate** current prompt (TEDS score)
2. **Analyze** cell-level errors on low-scoring images
3. **Meta-prompt** Gemini to suggest an improved fixer prompt
4. **Repeat** until TEDS >= 0.97 or plateau

Then validate best prompt on held-out test images and compare against UniTable-only baseline.

In [ ]:
import json
import re
import sys
import random
import base64
import io
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
from PIL import Image
from IPython.display import display, HTML
from bs4 import BeautifulSoup

# Project paths
project_root = Path.cwd().parent
huma_root = project_root / "Huma-Huma"
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(huma_root) not in sys.path:
    sys.path.insert(0, str(huma_root))
unitable_repo = huma_root / "unitable"
if str(unitable_repo) not in sys.path:
    sys.path.insert(0, str(unitable_repo))
teds_src = project_root / "pubtabnet" / "src"
if str(teds_src) not in sys.path:
    sys.path.insert(0, str(teds_src))

from metric import TEDS
from shared.client import client, DEFAULT_MODEL

# Config
TRAIN_SIZE = 3
MAX_ITERATIONS = 7
TARGET_TEDS = 0.97
MIN_IMPROVEMENT = 0.005

teds_scorer = TEDS(structure_only=False)
teds_structure = TEDS(structure_only=True)

print(f"Model: {DEFAULT_MODEL}")
print(f"Train size: {TRAIN_SIZE}, Max iterations: {MAX_ITERATIONS}, Target TEDS: {TARGET_TEDS}")
print("TEDS scorer ready")

## Load PubTabNet ground truth

In [ ]:
def format_html_clean(img):
    """Build GT HTML from PubTabNet annotation without prettify whitespace."""
    html_string = '<html><body><table>%s</table></body></html>' % ''.join(img['html']['structure']['tokens'])
    cell_nodes = list(re.finditer(r'(<td[^<>]*>)(</td>)', html_string))
    cells = [''.join(c['tokens']) for c in img['html']['cells']]
    offset = 0
    for n, cell in zip(cell_nodes, cells):
        html_string = html_string[:n.end(1) + offset] + cell + html_string[n.start(2) + offset:]
        offset += len(cell)
    return html_string

gt_path = project_root / "pubtabnet" / "examples" / "PubTabNet_Examples.jsonl"
img_dir = project_root / "pubtabnet" / "examples"

examples = []
with open(gt_path) as f:
    for line in f:
        examples.append(json.loads(line))

gt_data = []
for ex in examples:
    fname = ex["filename"]
    img_path = img_dir / fname
    if img_path.exists():
        gt_data.append({"filename": fname, "image_path": img_path, "gt_html": format_html_clean(ex)})

print(f"Loaded {len(gt_data)} images with ground truth")

## Train / test split

In [ ]:
random.seed(42)
random.shuffle(gt_data)
train_data = gt_data[:TRAIN_SIZE]
test_data = gt_data[TRAIN_SIZE:TRAIN_SIZE + 1]  # 1 held-out image for now

print(f"Train: {len(train_data)} images")
print(f"  " + ", ".join(d["filename"] for d in train_data))
print(f"Test:  {len(test_data)} images")
print(f"  " + ", ".join(d["filename"] for d in test_data))

## Load UniTable (local MPS)

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "unitable_extractor",
    str(huma_root / "table_extraction" / "unitable.py")
)
unitable_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(unitable_mod)

extractor = unitable_mod.UniTableExtractor()  # auto-detects MPS
print(f"UniTable device: {extractor.device}")

## Pre-extract UniTable HTML for all images
Run UniTable once and cache results — deterministic and expensive (~70s/image on MPS).

In [ ]:
all_data = train_data + test_data
unitable_cache = {}  # filename → html

for i, item in enumerate(all_data):
    fname = item["filename"]
    print(f"  [{i+1}/{len(all_data)}] UniTable: {fname} ... ", end="", flush=True)
    image = Image.open(item["image_path"]).convert("RGB")
    t0 = time.time()
    try:
        html_out = extractor.extract_html(image)
        elapsed = time.time() - t0
        unitable_cache[fname] = html_out
        print(f"{elapsed:.1f}s")
    except Exception as e:
        print(f"FAILED: {e}")
        unitable_cache[fname] = ""

success = sum(1 for v in unitable_cache.values() if v)
print(f"\nUniTable: {success}/{len(all_data)} extracted and cached")

## Hybrid extraction function (parameterized prompt)

In [ ]:
def hybrid_extract(image: Image.Image, unitable_html: str, fixer_prompt: str) -> str:
    """Run Gemini fixer with a parameterized prompt. Prompt must contain {html} placeholder."""
    if not unitable_html:
        return ""
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    img_bytes = buf.getvalue()
    prompt = fixer_prompt.format(html=unitable_html)

    response = client.models.generate_content(
        model=DEFAULT_MODEL,
        contents=[
            {"inline_data": {"mime_type": "image/png", "data": base64.b64encode(img_bytes).decode()}},
            prompt,
        ],
    )
    text = response.text.strip()
    text = re.sub(r'^```html\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    return text

print("hybrid_extract() ready")

## Evaluation function

In [ ]:
def evaluate_prompt(fixer_prompt: str, data: list, unitable_cache: dict) -> dict:
    """Run hybrid extraction with given prompt on data, return TEDS scores + per-image details."""
    per_image = []
    for item in data:
        fname = item["filename"]
        gt_html = item["gt_html"]
        ut_html = unitable_cache.get(fname, "")
        image = Image.open(item["image_path"]).convert("RGB")

        try:
            pred_html = hybrid_extract(image, ut_html, fixer_prompt)
        except Exception as e:
            print(f"  ERROR on {fname}: {e}")
            pred_html = ""

        teds_full = teds_scorer.evaluate(pred_html, gt_html) if pred_html else 0.0
        teds_struct = teds_structure.evaluate(pred_html, gt_html) if pred_html else 0.0

        per_image.append({
            "filename": fname,
            "teds": teds_full,
            "struct_teds": teds_struct,
            "pred_html": pred_html,
            "gt_html": gt_html,
        })
        print(f"  {fname}: TEDS={teds_full:.3f}, Struct={teds_struct:.3f}")

    mean_teds = sum(r["teds"] for r in per_image) / len(per_image) if per_image else 0.0
    mean_struct = sum(r["struct_teds"] for r in per_image) / len(per_image) if per_image else 0.0

    return {
        "mean_teds": mean_teds,
        "mean_struct_teds": mean_struct,
        "per_image": per_image,
    }

print("evaluate_prompt() ready")

## Error analysis function

In [ ]:
def analyze_errors(eval_result: dict, threshold: float = 0.97) -> str:
    """Compare pred vs GT HTML for low-TEDS images, extract cell-by-cell text diffs."""
    error_lines = []

    for img_result in eval_result["per_image"]:
        if img_result["teds"] >= threshold:
            continue

        fname = img_result["filename"]
        error_lines.append(f"\n### {fname} (TEDS={img_result['teds']:.3f})")

        pred_soup = BeautifulSoup(img_result["pred_html"], "html.parser")
        gt_soup = BeautifulSoup(img_result["gt_html"], "html.parser")

        pred_cells = [td.get_text(strip=True) for td in pred_soup.find_all("td")]
        gt_cells = [td.get_text(strip=True) for td in gt_soup.find_all("td")]

        # Compare cell counts
        if len(pred_cells) != len(gt_cells):
            error_lines.append(f"  Cell count mismatch: predicted {len(pred_cells)}, ground truth {len(gt_cells)}")

        # Show cell-level diffs (up to 15 per image)
        n_compare = min(len(pred_cells), len(gt_cells))
        diffs_shown = 0
        for i in range(n_compare):
            if pred_cells[i] != gt_cells[i] and diffs_shown < 15:
                error_lines.append(f"  Cell {i}: predicted='{pred_cells[i]}' | gt='{gt_cells[i]}'")
                diffs_shown += 1

        # Show extra/missing cells
        if len(pred_cells) > len(gt_cells):
            for i in range(len(gt_cells), min(len(pred_cells), len(gt_cells) + 5)):
                error_lines.append(f"  Extra cell {i}: '{pred_cells[i]}'")
        elif len(gt_cells) > len(pred_cells):
            for i in range(len(pred_cells), min(len(gt_cells), len(pred_cells) + 5)):
                error_lines.append(f"  Missing cell {i}: gt='{gt_cells[i]}'")

    if not error_lines:
        return "All images meet the TEDS threshold. No errors to report."

    return "\n".join(error_lines)

print("analyze_errors() ready")

## Meta-prompt + prompt history

In [ ]:
META_PROMPT = """You are a prompt engineering expert improving an OCR correction prompt.

## Current Prompt
{current_prompt}

## Performance
Mean TEDS: {mean_teds:.3f} (target: 0.97+)

## Cell-level errors found
{error_analysis}

## Task
Write an improved version of the prompt that addresses these specific errors.
Rules:
- Keep HTML output format (the model returns a corrected HTML table)
- The prompt must contain {{html}} as placeholder for the UniTable HTML input
- The model receives the table image alongside this prompt
- Keep the prompt under 500 words
- Focus on fixing the specific error patterns shown above
- Return ONLY the new prompt text, nothing else
"""

# Prompt history tracker
prompt_history = []

def record_iteration(iteration, prompt, eval_result):
    prompt_history.append({
        "iteration": iteration,
        "prompt": prompt,
        "mean_teds": eval_result["mean_teds"],
        "mean_struct_teds": eval_result["mean_struct_teds"],
        "prompt_length": len(prompt.split()),
        "timestamp": datetime.now().isoformat(),
    })

def show_history():
    df = pd.DataFrame(prompt_history)
    cols = ["iteration", "mean_teds", "mean_struct_teds", "prompt_length", "timestamp"]
    display(df[cols])

print("Meta-prompt and history tracker ready")

## Seed prompt

In [ ]:
current_prompt = """I have an HTML table extracted by a model from the attached image.
The table STRUCTURE (rows, columns, spanning) is correct, but some cell text may have OCR errors.

Look at the image and fix ONLY the cell text content. Do NOT change:
- Table structure (rows, columns, colspan, rowspan)
- HTML tags or attributes
- The number of cells

Return the corrected HTML table. Output ONLY the HTML, nothing else.

Current HTML:
{html}
"""

print(f"Seed prompt: {len(current_prompt.split())} words")
print(current_prompt[:200] + "...")

## Training loop

In [ ]:
for iteration in range(MAX_ITERATIONS):
    print(f"\n{'='*70}")
    print(f"ITERATION {iteration}")
    print(f"{'='*70}")

    # 1. Evaluate current prompt on train set
    print(f"\nEvaluating prompt ({len(current_prompt.split())} words)...")
    eval_result = evaluate_prompt(current_prompt, train_data, unitable_cache)
    mean_teds = eval_result["mean_teds"]
    print(f"\n  Mean TEDS: {mean_teds:.3f} | Mean Struct: {eval_result['mean_struct_teds']:.3f}")

    # 2. Record
    record_iteration(iteration, current_prompt, eval_result)

    # 3. Check stopping: target met
    if mean_teds >= TARGET_TEDS:
        print(f"\n  TARGET REACHED ({mean_teds:.3f} >= {TARGET_TEDS}). Stopping.")
        break

    # 4. Check plateau
    if iteration > 0:
        prev_teds = prompt_history[-2]["mean_teds"]
        delta = mean_teds - prev_teds
        print(f"  Delta from previous: {delta:+.4f}")
        if delta < MIN_IMPROVEMENT:
            print(f"\n  PLATEAU (delta {delta:.4f} < {MIN_IMPROVEMENT}). Stopping.")
            break

    # 5. Analyze errors
    print("\nAnalyzing errors...")
    error_analysis = analyze_errors(eval_result)
    print(error_analysis[:500])

    # 6. Call Gemini meta-prompt to generate improved prompt
    print("\nGenerating improved prompt via meta-prompt...")
    meta_filled = META_PROMPT.format(
        current_prompt=current_prompt,
        mean_teds=mean_teds,
        error_analysis=error_analysis,
    )
    response = client.models.generate_content(
        model=DEFAULT_MODEL,
        contents=[meta_filled],
    )
    new_prompt = response.text.strip()
    # Strip markdown fences if present
    new_prompt = re.sub(r'^```[a-z]*\s*', '', new_prompt)
    new_prompt = re.sub(r'\s*```$', '', new_prompt)

    # Validate the new prompt has {html} placeholder
    if "{html}" not in new_prompt:
        print("  WARNING: New prompt missing {html} placeholder. Appending it.")
        new_prompt += "\n\nCurrent HTML:\n{html}"

    print(f"  New prompt: {len(new_prompt.split())} words")
    current_prompt = new_prompt

print(f"\n{'='*70}")
print("TRAINING COMPLETE")
print(f"{'='*70}")
show_history()

## Prompt evolution display

In [ ]:
for entry in prompt_history:
    print(f"\n{'='*70}")
    print(f"Iteration {entry['iteration']} | TEDS: {entry['mean_teds']:.3f} | Words: {entry['prompt_length']}")
    print(f"{'='*70}")
    print(entry["prompt"][:600])
    if len(entry["prompt"]) > 600:
        print("...")

## Validate best prompt on held-out test set

In [ ]:
# Pick the best prompt from training
best_entry = max(prompt_history, key=lambda x: x["mean_teds"])
best_prompt = best_entry["prompt"]
print(f"Best prompt from iteration {best_entry['iteration']} (train TEDS: {best_entry['mean_teds']:.3f})")

# Evaluate on held-out test set
print(f"\nValidating on {len(test_data)} test images...")
test_result = evaluate_prompt(best_prompt, test_data, unitable_cache)

print(f"\n{'='*60}")
print(f"TEST SET RESULTS")
print(f"{'='*60}")
print(f"  Mean TEDS:      {test_result['mean_teds']:.3f}")
print(f"  Mean Struct:    {test_result['mean_struct_teds']:.3f}")
print(f"  Train TEDS:     {best_entry['mean_teds']:.3f}")
print(f"  Generalization: {'PASS' if test_result['mean_teds'] >= TARGET_TEDS else 'BELOW TARGET'}")

## Baseline comparison: UniTable-only vs best hybrid

In [ ]:
# Score UniTable-only (no Gemini fixer) against GT on test set
rows = []
for item in test_data:
    fname = item["filename"]
    gt_html = item["gt_html"]
    ut_html = unitable_cache.get(fname, "")

    ut_teds = teds_scorer.evaluate(ut_html, gt_html) if ut_html else 0.0
    ut_struct = teds_structure.evaluate(ut_html, gt_html) if ut_html else 0.0

    # Find hybrid result from test_result
    hybrid_entry = next((r for r in test_result["per_image"] if r["filename"] == fname), None)
    hy_teds = hybrid_entry["teds"] if hybrid_entry else 0.0
    hy_struct = hybrid_entry["struct_teds"] if hybrid_entry else 0.0

    rows.append({
        "filename": fname,
        "UniTable TEDS": round(ut_teds, 3),
        "UniTable Struct": round(ut_struct, 3),
        "Hybrid TEDS": round(hy_teds, 3),
        "Hybrid Struct": round(hy_struct, 3),
        "Delta": round(hy_teds - ut_teds, 3),
    })

df_compare = pd.DataFrame(rows)
display(df_compare)

# Aggregates
ut_mean = df_compare["UniTable TEDS"].mean()
hy_mean = df_compare["Hybrid TEDS"].mean()
print(f"\nTest set aggregates:")
print(f"  UniTable-only:  {ut_mean:.3f} mean TEDS")
print(f"  Best hybrid:    {hy_mean:.3f} mean TEDS")
print(f"  Improvement:    {hy_mean - ut_mean:+.3f}")

## Summary chart: TEDS per iteration

In [ ]:
import matplotlib.pyplot as plt

iterations = [e["iteration"] for e in prompt_history]
teds_scores = [e["mean_teds"] for e in prompt_history]
struct_scores = [e["mean_struct_teds"] for e in prompt_history]

# UniTable baseline on train set
ut_baseline_scores = []
for item in train_data:
    ut_html = unitable_cache.get(item["filename"], "")
    score = teds_scorer.evaluate(ut_html, item["gt_html"]) if ut_html else 0.0
    ut_baseline_scores.append(score)
ut_baseline = sum(ut_baseline_scores) / len(ut_baseline_scores)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(iterations, teds_scores, 'o-', color='#2196F3', linewidth=2, markersize=8, label='Hybrid TEDS')
ax.plot(iterations, struct_scores, 's--', color='#4CAF50', linewidth=1.5, markersize=6, label='Hybrid Struct TEDS')
ax.axhline(y=ut_baseline, color='#FF9800', linestyle=':', linewidth=2, label=f'UniTable baseline ({ut_baseline:.3f})')
ax.axhline(y=TARGET_TEDS, color='#F44336', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Target ({TARGET_TEDS})')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('TEDS Score', fontsize=12)
ax.set_title('Fixer Prompt Training: TEDS per Iteration', fontsize=14)
ax.legend(fontsize=10)
ax.set_ylim(min(min(teds_scores), ut_baseline) - 0.02, 1.01)
ax.set_xticks(iterations)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Export best prompt

In [ ]:
print("="*70)
print("BEST FIXER PROMPT")
print(f"From iteration {best_entry['iteration']} | Train TEDS: {best_entry['mean_teds']:.3f} | Test TEDS: {test_result['mean_teds']:.3f}")
print("="*70)
print()
print(best_prompt)